# 06 · Observability

The scaffold's Terraform set up three layers of telemetry, and the agents have been writing to them since
notebook 04:

- **Cloud Trace:** one span per agent, model call and tool call;
- **prompt/response logging:** JSONL files in Cloud Storage, exposed as a BigQuery view;
- **BigQuery Agent Analytics:** one structured row per agent event.

Run notebook 05 first, so there is something to look at.

In [ ]:
from nbsetup import *

In [ ]:
console("traces/explorer")

In [ ]:
console(f"vertex-ai/agents/agent-engines/locations/{REGION}/agent-engines/{engine().rsplit('/', 1)[-1]}")

## One trace across both services

The orchestrator runs on Agent Runtime and the Compliance Agent on Cloud Run, but the trace context travels with
the A2A call, so a single trace covers both. This cell finds the richest recent orchestrator trace and counts its
spans by kind.

In [ ]:
import collections, datetime
import google.auth, google.auth.transport.requests, requests

creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
creds.refresh(google.auth.transport.requests.Request())
since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
traces = requests.get(
    f"https://cloudtrace.googleapis.com/v1/projects/{PROJECT}/traces",
    params={"startTime": since, "pageSize": 20, "view": "COMPLETE", "filter": "span:invoke_agent"},
    headers={"Authorization": f"Bearer {creds.token}"},
).json().get("traces", [])
if not traces:
    raise SystemExit("No traces in the last day: run notebook 05 first.")
root = [t for t in traces if any(s["name"] == "invoke_agent growth_risk_orchestrator" for s in t["spans"])]
t = max(root or traces, key=lambda t: len(t["spans"]))
kinds = collections.Counter(" ".join(s["name"].split()[:2]) for s in t["spans"])
print(f"trace {t['traceId'][:12]}: {len(t['spans'])} spans")
for name, n in sorted(kinds.items()):
    if name.startswith(("invoke_agent", "execute_tool", "call_llm", "a2a", "POST")):
        print(f"  {n:3} {name}")

## BigQuery Agent Analytics

One row per agent event: model calls, tool calls, state changes. Plain SQL, so you can build your own dashboards.

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty "
SELECT agent, event_type, COUNT(*) AS events
FROM \`$PROJECT.growth_risk_agent_telemetry.agent_events\`
WHERE timestamp > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
GROUP BY 1, 2 ORDER BY 1, 3 DESC LIMIT 25"

**What to notice**

- `OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT=NO_CONTENT` keeps prompts out of the *spans*, but prompts and responses still go to Cloud Storage and the BigQuery `completions_view`. Treat that bucket as sensitive.
- Expected noise: under Agent Identity, `gcsfs` logs "Could not determine bucket type … 401" and falls back; uploads still succeed. The Analytics plugin's own BigQuery calls show up as separate root traces.

Next: [07 · Evaluation](07_evaluation.ipynb).